# PLAPT ablation — Stage 2: compute the frozen embeddings

This is the only step that needs a GPU. It runs ProtBERT over ~8.6k unique protein
sequences and ChemBERTa over ~182k unique ligands, and caches the resulting
vectors so every later experiment is pure CPU arithmetic.

**Before you run anything:** Runtime → Change runtime type → **L4 GPU**.

Expected wall time on an L4: ~3 min for the ligands, ~10-15 min for the proteins.

In [ ]:
# Confirm you actually got a GPU. Colab will happily hand you a CPU runtime and
# this script would still "work" -- just 100x slower. Always check.
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

## 1. Install dependencies

Colab ships torch already. We only need `transformers`. Pinning the version is
good practice: a silent major-version bump is a classic source of "it worked
last week" failures.

In [ ]:
!pip -q install "transformers>=4.44,<6"
import torch, transformers
print("torch", torch.__version__, "| transformers", transformers.__version__)
print("cuda available:", torch.cuda.is_available())

## 2. Upload the four files from your laptop

Drag these into the upload prompt below (they total ~11 MB):

- `scripts/02_embed.py`
- `data/processed/proteins.parquet`
- `data/processed/ligands.parquet`
- `data/processed/pairs.parquet`  *(not used by this stage, but handy to have here)*

> Uploading files by hand is fine for one run, but it is the part of this
> workflow that does not scale. The grown-up version is to `git push` the repo
> and `!git clone` it here, so the notebook and the code can never drift out of
> sync. Worth graduating to once the experiment settles down.

In [ ]:
from google.colab import files
up = files.upload()
print()
for name in up: print(" uploaded:", name)

## 3. Mount Drive for the output

The embeddings come to roughly 315 MB (proteins fp32, ligands fp16 -- see the
script's docstring for why that asymmetry is a scientific choice, not a storage one). Browser downloads that size are flaky and
a Colab runtime is deleted the moment it disconnects, so write the results to
Drive instead — then they survive, and you can pull them down once at your
leisure.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

OUT = '/content/drive/MyDrive/plapt-experiment/emb'
import os; os.makedirs(OUT, exist_ok=True)
print("writing embeddings to:", OUT)

## 4. Run it

The script prints a running throughput and ETA. Two things worth watching:

- The **ligand** pass should fly (short strings, a small 6-layer model).
- The **protein** pass starts fast and slows down, because the script sorts by
  sequence length and the long sequences are at the end. Attention cost grows
  with the *square* of length, so the last few hundred proteins can take as long
  as the first several thousand. That slowdown is expected, not a hang.

In [ ]:
!python 02_embed.py --data . --out "$OUT"

## 5. Check the output and copy the small files too

The final sanity print is the one that matters. If `avg per-dim std across
inputs` were near zero for either encoder, it would mean the model is emitting
essentially the same vector for every input — and the entire downstream
experiment would be measuring nothing. Confirm it is comfortably non-zero.

In [ ]:
import shutil, os, numpy as np
for f in ['proteins.parquet','ligands.parquet','pairs.parquet','meta.json']:
    if os.path.exists(f): shutil.copy(f, OUT)

for f in sorted(os.listdir(OUT)):
    p = os.path.join(OUT, f)
    print(f"{os.path.getsize(p)/1e6:9.1f} MB  {f}")

# The script already printed this, but confirm it on the full table yourself.
# "Effective rank" = how many independent directions the representation really
# uses. ~1.0 means every input collapses onto a single line.
print()
for name in ['prot_pooler','prot_mean','mol_pooler','mol_mean']:
    a = np.load(os.path.join(OUT, name + '.npy')).astype('float64')
    if len(a) > 4000:
        a = a[np.random.default_rng(0).choice(len(a), 4000, replace=False)]
    s = np.linalg.svd(a - a.mean(0), compute_uv=False)
    q = s**2 / (s**2).sum()
    eff = float(np.exp(-(q*np.log(q+1e-300)).sum()))
    print(f"  {name:12s} effective rank {eff:7.1f} / {a.shape[1]}"
          f"   dims for 90% of variance: {int(np.searchsorted(np.cumsum(q),0.90)+1)}")

## 6. Pull the results to your laptop

Open [drive.google.com](https://drive.google.com), find `plapt-experiment/emb`,
and download the folder into `C:\Coding\plapt-experiment\data\processed\`.

You need these six files for stage 3:

```
prot_pooler.npy    mol_pooler.npy     <- what PLAPT actually uses (fp32 / fp16)
prot_mean.npy      mol_mean.npy       <- the free upgrade, for a later experiment
embed_meta.json    (+ the parquets you already have locally)
```